# 08_3_2 DQA Scene Phase2 FedSTO/DQA Sweep

目的は **FedSTO phase2 を超えるために、phase2 だけを切り出して比較する**ことです。08_3 では `e_r02` が phase1 best を少し超えた一方、長く回すと落ちました。ここでは「FedSTO と同じ phase2」を必ず 1 本入れ、残り 4 本で DQA らしい品質ゲート・server anchor・client drift 抑制を試します。

根拠:

- FedSTO は phase1 で backbone を選択的に更新し、phase2 で full-parameter training + orthogonal regularization を使います。論文では phase2 が non-backbone predictor のバイアスを抑えて robustness を上げる役割として説明されています: https://arxiv.org/abs/2310.17097
- SSOD では class confidence だけでは localization quality を保証できないため、pseudo label の品質推定・動的閾値・loss reweight が重要です: https://arxiv.org/abs/2106.00168
- PseCo も noisy pseudo boxes に対して、localization quality / consistency を見る必要があると述べています: https://arxiv.org/abs/2203.16317
- Federated semi-supervised learning では non-IID client の信頼性差が問題になり、単純 FedAvg より client update の頑健な集約が必要になりえます: https://openaccess.thecvf.com/content/CVPR2022/papers/Liang_RSCFed_Random_Sampling_Consensus_Federated_Semi-Supervised_Learning_CVPR_2022_paper.pdf

仮説: **DQA が伸びない原因は phase2 で head/bbox/cls を守りすぎたこと**。ただし FedSTO をそのまま真似すると noisy pseudoGT に引っ張られるので、DQA は `full update を許可しつつ、quality と server anchor で update を絞る` 方向で勝ちに行きます。

In [1]:
from __future__ import annotations

import json
import os
import signal
import shutil
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "dynamic_quality_aware_classwise_aggregation").exists() and (path / "navigating_data_heterogeneity").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")

REPO_ROOT = find_repo_root(Path.cwd())

DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2_phase2_fedsto_dqa_sweep.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_scene_protocol.py"
SOURCE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_scene_tri_stage_policy_8h"
BASE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_3_2_phase2_fedsto_dqa_sweep"
BASE_STATS_ROOT = DQA_ROOT / "stats_dqa08_3_2_phase2_fedsto_dqa_sweep"
BASE_LOG_ROOT = DQA_ROOT / "logs_dqa08_3_2_phase2_fedsto_dqa_sweep"

for path in [RUN_SCRIPT, EVAL_SCRIPT, SOURCE_WORK_ROOT]:
    print(path, "exists=", path.exists())

BASE_WORK_ROOT.mkdir(parents=True, exist_ok=True)
BASE_STATS_ROOT.mkdir(parents=True, exist_ok=True)
BASE_LOG_ROOT.mkdir(parents=True, exist_ok=True)

print("workspace:", BASE_WORK_ROOT)
print("stats:", BASE_STATS_ROOT)
print("logs:", BASE_LOG_ROOT)

/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase2_fedsto_dqa_sweep.py exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h exists= True
workspace: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_2_phase2_fedsto_dqa_sweep
stats: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_3_2_phase2_fedsto_dqa_sweep
logs: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_2_phase2_fedsto_dqa_sweep


## Variant Design

5 本とも phase2-only で、seed は 08 phase1 best 候補の `phase1_round003_global.pt` に固定します。各 10 round にして、08_3 と同じくらいの時間で終わる想定です。

- `a_fedsto_exact_r003`: FedSTO phase2 と同じ full-parameter + bbox/cls pseudoGT + FedAvg。比較基準。
- `b_fedsto_dqa_residual_r003`: client 学習は FedSTO のまま、集約だけ DQA residual/server anchor に変更。FedSTO を壊さず DQA が勝てるか確認。
- `c_dqa_high_light_head_r003`: high-quality pseudoGT だけ bbox/cls を許可。08_3 の head-protected から一歩攻める。
- `d_dqa_strict_localization_r003`: high threshold をさらに上げ、bbox/cls loss を薄くして mAP50:95 を狙う。
- `e_dqa_consensus_anchor_r003`: federated non-IID 対策として client drift をかなり抑える。RSCFed 的な「信頼できない client update を集約で薄める」方向。

In [2]:
PHASE2_ROUNDS_PER_VARIANT = 10
BATCH_SIZE = 160
WORKERS = 10
GPUS = 2
MASTER_PORT_BASE = 29720
STREAM_TRAIN_OUTPUT = False
MIN_FREE_GIB = 30

# 空なら全 variant 実行。必要なら ["a_fedsto_exact_r003"] のように絞る。
SELECTED_VARIANTS: list[str] = []

DEFAULT_POLICY = {
    "adapt_start_round": 2,
    "min_low": 0.38,
    "max_low": 0.48,
    "min_mid": 0.58,
    "max_mid": 0.72,
    "min_high": 0.84,
    "max_high": 0.92,
    "low_shift": 0.02,
    "high_shift": 0.05,
    "mid_gap": 0.20,
    "high_mid_gap": 0.18,
    "min_high_gap": 0.16,
    "max_nms": 0.45,
    "teacher_min": 0.28,
    "rare_count": 250.0,
    "rare_max_low": 0.43,
    "rare_max_mid": 0.66,
    "rare_max_high": 0.89,
    "low_step_limit": 0.02,
    "mid_step_limit": 0.025,
    "high_step_limit": 0.035,
    "nms_step_limit": 0.02,
    "low_mid_obj_weight": 0.35,
    "mid_high_obj_weight": 0.90,
}

VARIANTS = [
    {
        "name": "a_fedsto_exact_r003",
        "description": "FedSTO phase2 exact control: full parameter client/server, original SSOD thresholds/losses, plain FedAvg.",
        "source_phase1_round": 3,
        "profile": "fedsto_exact",
        "dqa_start_phase": 99,
        "client_train_scope": "all",
        "client_lr0": 0.01,
        "server_lr0": 0.01,
        "client_orthogonal": 1e-4,
        "server_orthogonal": 1e-4,
        "classwise_blend": 0.35,
        "server_anchor": 1.25,
        "temperature": 2.0,
        "stability_lambda": 0.4,
        "residual_start": 0.35,
        "residual_end": 0.35,
        "min_server_alpha_start": 0.0,
        "min_server_alpha_end": 0.0,
        "ssod": {},
        "policy": {},
    },
    {
        "name": "b_fedsto_dqa_residual_r003",
        "description": "FedSTO client objective + DQA server-anchored residual aggregation.",
        "source_phase1_round": 3,
        "profile": "fedsto_exact",
        "dqa_start_phase": 2,
        "client_train_scope": "all",
        "client_lr0": 0.01,
        "server_lr0": 0.01,
        "client_orthogonal": 1e-4,
        "server_orthogonal": 1e-4,
        "classwise_blend": 0.16,
        "server_anchor": 4.0,
        "temperature": 2.0,
        "stability_lambda": 0.35,
        "residual_start": 0.22,
        "residual_end": 0.08,
        "min_server_alpha_start": 0.55,
        "min_server_alpha_end": 0.65,
        "ssod": {},
        "policy": {},
    },
    {
        "name": "c_dqa_high_light_head_r003",
        "description": "DQA learned thresholds, high pseudoGT gets bbox/cls, lower LR and moderate residual.",
        "source_phase1_round": 3,
        "profile": "dqa_high_light",
        "dqa_start_phase": 2,
        "client_train_scope": "all",
        "client_lr0": 0.001,
        "server_lr0": 0.003,
        "client_orthogonal": 1e-4,
        "server_orthogonal": 1e-4,
        "classwise_blend": 0.08,
        "server_anchor": 8.0,
        "temperature": 2.2,
        "stability_lambda": 0.45,
        "residual_start": 0.12,
        "residual_end": 0.04,
        "min_server_alpha_start": 0.72,
        "min_server_alpha_end": 0.78,
        "ssod": {
            "DQA0832_BOX_LOSS_WEIGHT": 0.010,
            "DQA0832_OBJ_LOSS_WEIGHT": 0.34,
            "DQA0832_CLS_LOSS_WEIGHT": 0.040,
            "DQA0832_LOW_MID_OBJ_WEIGHT": 0.30,
            "DQA0832_MID_HIGH_OBJ_WEIGHT": 0.80,
            "DQA0832_IGNORE_OBJ": 0,
            "DQA0832_PSEUDO_LABEL_WITH_BBOX": 1,
            "DQA0832_PSEUDO_LABEL_WITH_CLS": 1,
        },
        "policy": {},
    },
    {
        "name": "d_dqa_strict_localization_r003",
        "description": "Stricter high threshold, very light bbox/cls, designed to improve localization AP without recall spam.",
        "source_phase1_round": 3,
        "profile": "dqa_high_light",
        "dqa_start_phase": 2,
        "client_train_scope": "all",
        "client_lr0": 0.0008,
        "server_lr0": 0.003,
        "client_orthogonal": 1e-4,
        "server_orthogonal": 1e-4,
        "classwise_blend": 0.06,
        "server_anchor": 12.0,
        "temperature": 2.5,
        "stability_lambda": 0.55,
        "residual_start": 0.10,
        "residual_end": 0.03,
        "min_server_alpha_start": 0.80,
        "min_server_alpha_end": 0.86,
        "ssod": {
            "DQA0832_BOX_LOSS_WEIGHT": 0.006,
            "DQA0832_OBJ_LOSS_WEIGHT": 0.30,
            "DQA0832_CLS_LOSS_WEIGHT": 0.025,
            "DQA0832_LOW_MID_OBJ_WEIGHT": 0.20,
            "DQA0832_MID_HIGH_OBJ_WEIGHT": 0.65,
            "DQA0832_IGNORE_OBJ": 0,
            "DQA0832_PSEUDO_LABEL_WITH_BBOX": 1,
            "DQA0832_PSEUDO_LABEL_WITH_CLS": 1,
        },
        "policy": {
            "min_low": 0.42,
            "max_low": 0.50,
            "min_mid": 0.66,
            "max_mid": 0.78,
            "min_high": 0.90,
            "max_high": 0.96,
            "rare_max_low": 0.45,
            "rare_max_mid": 0.70,
            "rare_max_high": 0.92,
            "max_nms": 0.48,
            "high_shift": 0.07,
        },
    },
    {
        "name": "e_dqa_consensus_anchor_r003",
        "description": "RSCFed-inspired conservative aggregation: allow full client learning but strongly limit drift into global.",
        "source_phase1_round": 3,
        "profile": "dqa_high_light",
        "dqa_start_phase": 2,
        "client_train_scope": "all",
        "client_lr0": 0.001,
        "server_lr0": 0.003,
        "client_orthogonal": 1e-4,
        "server_orthogonal": 1e-4,
        "classwise_blend": 0.04,
        "server_anchor": 16.0,
        "temperature": 3.0,
        "stability_lambda": 0.65,
        "residual_start": 0.07,
        "residual_end": 0.02,
        "min_server_alpha_start": 0.86,
        "min_server_alpha_end": 0.90,
        "ssod": {
            "DQA0832_BOX_LOSS_WEIGHT": 0.008,
            "DQA0832_OBJ_LOSS_WEIGHT": 0.28,
            "DQA0832_CLS_LOSS_WEIGHT": 0.030,
            "DQA0832_LOW_MID_OBJ_WEIGHT": 0.20,
            "DQA0832_MID_HIGH_OBJ_WEIGHT": 0.55,
            "DQA0832_IGNORE_OBJ": 0,
            "DQA0832_PSEUDO_LABEL_WITH_BBOX": 1,
            "DQA0832_PSEUDO_LABEL_WITH_CLS": 1,
        },
        "policy": {
            "min_high": 0.86,
            "max_high": 0.94,
            "min_mid": 0.62,
            "max_mid": 0.74,
            "high_shift": 0.05,
        },
    },
]

selected = [v for v in VARIANTS if not SELECTED_VARIANTS or v["name"] in SELECTED_VARIANTS]
pd.DataFrame([
    {
        "name": v["name"],
        "profile": v["profile"],
        "dqa_start": v["dqa_start_phase"],
        "scope": v["client_train_scope"],
        "lr": v["client_lr0"],
        "residual": f"{v['residual_start']}->{v['residual_end']}",
        "server_alpha": f"{v['min_server_alpha_start']}->{v['min_server_alpha_end']}",
        "description": v["description"],
    }
    for v in selected
])

,name,profile,dqa_start,scope,lr,residual,server_alpha,description
0,a_fedsto_exact_r003,fedsto_exact,99,all,0.0100,0.35->0.35,0.0->0.0,FedSTO phase2 exact control: full parameter cl...
1,b_fedsto_dqa_residual_r003,fedsto_exact,2,all,0.0100,0.22->0.08,0.55->0.65,FedSTO client objective + DQA server-anchored ...
2,c_dqa_high_light_head_r003,dqa_high_light,2,all,0.0010,0.12->0.04,0.72->0.78,"DQA learned thresholds, high pseudoGT gets bbo..."
3,d_dqa_strict_localization_r003,dqa_high_light,2,all,0.0008,0.1->0.03,0.8->0.86,"Stricter high threshold, very light bbox/cls, ..."
4,e_dqa_consensus_anchor_r003,dqa_high_light,2,all,0.0010,0.07->0.02,0.86->0.9,RSCFed-inspired conservative aggregation: allo...


In [3]:
def variant_work_root(variant: dict) -> Path:
    return BASE_WORK_ROOT / variant["name"]


def variant_stats_root(variant: dict) -> Path:
    return BASE_STATS_ROOT / variant["name"]


def variant_runner_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_runner.out"


def variant_train_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_train.log"


def variant_pid_path(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}.pid"


def policy_value(variant: dict, key: str):
    return dict(DEFAULT_POLICY, **variant.get("policy", {}))[key]


def variant_env(variant: dict) -> dict[str, str]:
    env = os.environ.copy()
    stats_root = variant_stats_root(variant)
    stats_root.mkdir(parents=True, exist_ok=True)

    env["DQA0832_VARIANT"] = variant["name"]
    env["DQA0832_PROFILE"] = variant["profile"]
    env["DQA0832_SOURCE_WORK_ROOT"] = str(SOURCE_WORK_ROOT)
    env["DQA0832_SOURCE_PHASE1_ROUND"] = str(variant["source_phase1_round"])
    env["DQA0832_CLIENT_TRAIN_SCOPE"] = variant["client_train_scope"]
    env["DQA0832_SERVER_TRAIN_SCOPE"] = "all"
    env["DQA0832_CLIENT_LR0"] = str(variant["client_lr0"])
    env["DQA0832_SERVER_LR0"] = str(variant["server_lr0"])
    env["DQA0832_CLIENT_ORTHOGONAL_WEIGHT"] = str(variant["client_orthogonal"])
    env["DQA0832_SERVER_ORTHOGONAL_WEIGHT"] = str(variant["server_orthogonal"])
    env["DQA0832_RESIDUAL_START"] = str(variant["residual_start"])
    env["DQA0832_RESIDUAL_END"] = str(variant["residual_end"])
    env["DQA0832_MIN_SERVER_ALPHA_START"] = str(variant["min_server_alpha_start"])
    env["DQA0832_MIN_SERVER_ALPHA_END"] = str(variant["min_server_alpha_end"])
    env["DQA0832_PHASE2_ROUNDS"] = str(PHASE2_ROUNDS_PER_VARIANT)
    env["DQA08_STATS_ROOT"] = str(stats_root.resolve())
    env["DQA08_THRESHOLD_LOG"] = str((stats_root / "phase2_fedsto_dqa_policy_schedule.jsonl").resolve())

    for key, value in variant.get("ssod", {}).items():
        env[key] = str(value)

    env["DQA08_ADAPT_START_ROUND"] = str(policy_value(variant, "adapt_start_round"))
    env["DQA08_MIN_LOW"] = str(policy_value(variant, "min_low"))
    env["DQA08_MAX_LOW"] = str(policy_value(variant, "max_low"))
    env["DQA08_MIN_MID"] = str(policy_value(variant, "min_mid"))
    env["DQA08_MAX_MID"] = str(policy_value(variant, "max_mid"))
    env["DQA08_MIN_HIGH"] = str(policy_value(variant, "min_high"))
    env["DQA08_MAX_HIGH"] = str(policy_value(variant, "max_high"))
    env["DQA08_LOW_SHIFT"] = str(policy_value(variant, "low_shift"))
    env["DQA08_HIGH_SHIFT"] = str(policy_value(variant, "high_shift"))
    env["DQA08_MID_GAP"] = str(policy_value(variant, "mid_gap"))
    env["DQA08_HIGH_MID_GAP"] = str(policy_value(variant, "high_mid_gap"))
    env["DQA08_MIN_HIGH_GAP"] = str(policy_value(variant, "min_high_gap"))
    env["DQA08_MAX_NMS"] = str(policy_value(variant, "max_nms"))
    env["DQA08_TEACHER_MIN"] = str(policy_value(variant, "teacher_min"))
    env["DQA08_RARE_COUNT"] = str(policy_value(variant, "rare_count"))
    env["DQA08_RARE_MAX_LOW"] = str(policy_value(variant, "rare_max_low"))
    env["DQA08_RARE_MAX_MID"] = str(policy_value(variant, "rare_max_mid"))
    env["DQA08_RARE_MAX_HIGH"] = str(policy_value(variant, "rare_max_high"))
    env["DQA08_LOW_STEP_LIMIT"] = str(policy_value(variant, "low_step_limit"))
    env["DQA08_MID_STEP_LIMIT"] = str(policy_value(variant, "mid_step_limit"))
    env["DQA08_HIGH_STEP_LIMIT"] = str(policy_value(variant, "high_step_limit"))
    env["DQA08_NMS_STEP_LIMIT"] = str(policy_value(variant, "nms_step_limit"))
    env.setdefault("DQA0832_LOW_MID_OBJ_WEIGHT", str(policy_value(variant, "low_mid_obj_weight")))
    env.setdefault("DQA0832_MID_HIGH_OBJ_WEIGHT", str(policy_value(variant, "mid_high_obj_weight")))
    return env


def train_cmd(variant: dict, *, stream: bool = STREAM_TRAIN_OUTPUT) -> list[str]:
    cmd = [
        sys.executable,
        str(RUN_SCRIPT),
        "--workspace-root",
        str(variant_work_root(variant)),
        "--stats-root",
        str(variant_stats_root(variant)),
        "--phase1-rounds",
        "0",
        "--phase2-rounds",
        str(PHASE2_ROUNDS_PER_VARIANT),
        "--batch-size",
        str(BATCH_SIZE),
        "--workers",
        str(WORKERS),
        "--gpus",
        str(GPUS),
        "--master-port",
        str(MASTER_PORT_BASE + selected.index(variant)),
        "--log-file",
        str(variant_train_log(variant)),
        "--classwise-blend",
        str(variant["classwise_blend"]),
        "--server-anchor",
        str(variant["server_anchor"]),
        "--temperature",
        str(variant["temperature"]),
        "--stability-lambda",
        str(variant["stability_lambda"]),
        "--dqa-start-phase",
        str(variant["dqa_start_phase"]),
        "--min-free-gib",
        str(MIN_FREE_GIB),
    ]
    if stream:
        cmd.append("--stream-train-output")
    return cmd


def history_rows(variant: dict) -> list[dict]:
    path = variant_work_root(variant) / "history.json"
    if not path.exists():
        return []
    return json.loads(path.read_text())


def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    try:
        return int(path.read_text().strip())
    except Exception:
        return None


def pid_state(pid: int | None) -> str:
    if pid is None:
        return "no pid"
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return "stopped"
    except PermissionError:
        return "running?"
    return "running"


def tail_lines(path: Path, n: int = 30) -> list[str]:
    if not path.exists():
        return [f"missing: {path}"]
    return path.read_text(errors="replace").splitlines()[-n:]

print("helpers ready")

helpers ready


In [4]:
# 実行セル。途中で止まっても history/global checkpoint があれば再利用します。
if not selected:
    raise RuntimeError("No selected variants")

for variant in selected:
    history = history_rows(variant)
    completed_phase2 = sum(1 for row in history if int(row.get("phase", 0)) == 2)
    print("\n===", variant["name"], "===")
    print(variant["description"])
    print(f"completed_phase2: {completed_phase2}/{PHASE2_ROUNDS_PER_VARIANT}")
    if completed_phase2 >= PHASE2_ROUNDS_PER_VARIANT:
        print("already completed")
        continue

    pid_path = variant_pid_path(variant)
    existing_pid = read_pid(pid_path)
    if pid_state(existing_pid) == "running":
        print("already running pid:", existing_pid)
        continue

    runner_log = variant_runner_log(variant)
    runner_log.parent.mkdir(parents=True, exist_ok=True)
    cmd = train_cmd(variant)
    env = variant_env(variant)
    print("cmd:", " ".join(cmd))
    print("runner_log:", runner_log)
    print("train_log:", variant_train_log(variant))

    with runner_log.open("a", encoding="utf-8", buffering=1) as log:
        log.write("\n" + "=" * 100 + "\n")
        log.write(" ".join(cmd) + "\n")
        process = subprocess.Popen(
            cmd,
            cwd=REPO_ROOT,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        pid_path.write_text(str(process.pid), encoding="utf-8")
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        rc = process.wait()
        if rc != 0:
            raise RuntimeError(f"{variant['name']} failed with exit code {rc}. See {runner_log}")
        print("variant completed:", variant["name"])



=== a_fedsto_exact_r003 ===
FedSTO phase2 exact control: full parameter client/server, original SSOD thresholds/losses, plain FedAvg.
completed_phase2: 10/10
already completed

=== b_fedsto_dqa_residual_r003 ===
FedSTO client objective + DQA server-anchored residual aggregation.
completed_phase2: 10/10
already completed

=== c_dqa_high_light_head_r003 ===
DQA learned thresholds, high pseudoGT gets bbox/cls, lower LR and moderate residual.
completed_phase2: 0/10
cmd: /root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase2_fedsto_dqa_sweep.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_2_phase2_fedsto_dqa_sweep/c_dqa_high_light_head_r003 --stats-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_3_2_phase2_fedsto_dqa_sweep/c_dqa_high_light_head_r003 --phase1-rounds 0 --phase2-rounds 10 --batch-size 160 --w

In [5]:
# 進捗確認セル
rows = []
for variant in selected:
    history = history_rows(variant)
    latest = Path(history[-1]["global"]) if history else variant_work_root(variant) / "global_checkpoints" / "round000_warmup.pt"
    rows.append({
        "variant": variant["name"],
        "pid": read_pid(variant_pid_path(variant)),
        "pid_state": pid_state(read_pid(variant_pid_path(variant))),
        "phase2": f"{sum(1 for row in history if int(row.get('phase', 0)) == 2)}/{PHASE2_ROUNDS_PER_VARIANT}",
        "latest": str(latest),
        "latest_exists": latest.exists(),
        "free_gib": round(shutil.disk_usage(variant_work_root(variant)).free / 1024**3, 2) if variant_work_root(variant).exists() else None,
    })

display(pd.DataFrame(rows))

for variant in selected:
    print("\n===", variant["name"], "runner tail ===")
    print("\n".join(tail_lines(variant_runner_log(variant), 20)))

,variant,pid,pid_state,phase2,latest,latest_exists,free_gib
0,a_fedsto_exact_r003,2434841,stopped,10/10,/app/Object_Detection/dynamic_quality_aware_cl...,True,101.75
1,b_fedsto_dqa_residual_r003,2521018,stopped,10/10,/app/Object_Detection/dynamic_quality_aware_cl...,True,101.75
2,c_dqa_high_light_head_r003,2646085,stopped,10/10,/app/Object_Detection/dynamic_quality_aware_cl...,True,101.75
3,d_dqa_strict_localization_r003,2735707,stopped,10/10,/app/Object_Detection/dynamic_quality_aware_cl...,True,101.75
4,e_dqa_consensus_anchor_r003,2825411,stopped,10/10,/app/Object_Detection/dynamic_quality_aware_cl...,True,101.75



=== a_fedsto_exact_r003 runner tail ===
Training output appended to /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_2_phase2_fedsto_dqa_sweep/a_fedsto_exact_r003_train.log
Completed DQA-CWA phase 2 round 8: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_2_phase2_fedsto_dqa_sweep/a_fedsto_exact_r003/global_checkpoints/phase2_round008_global.pt
/root/micromamba/envs/al_yolov8/bin/python -m torch.distributed.run --nproc_per_node 2 --master_port 29720 train.py --cfg /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_2_phase2_fedsto_dqa_sweep/a_fedsto_exact_r003/configs/runtime_phase2_client_round9_dqa_phase2_round009_client0_highway.yaml
Training output appended to /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_2_phase2_fedsto_dqa_sweep/a_fedsto_exact_r003_train.log
/root/micromamba/envs/al_yolov8/bin/python -m torch.distributed.run --nproc_per

In [6]:
# final checkpoint + early checkpoint 評価。
# phase2 は early peak が出やすいので r01/r02/r03/r10 をまず見る。
EVAL_WORKSPACE = variant_work_root(selected[0])
REPORT_DIR = EVAL_WORKSPACE / "validation_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

checkpoints: list[tuple[str, Path]] = []
seed03 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round003_global.pt"
seed12 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round012_global.pt"
old08_3_e02 = DQA_ROOT / "efficientteacher_dqa08_3_phase2_update_gating_sweep" / "e_ug_best_phase1_r003" / "global_checkpoints" / "phase2_round002_global.pt"
checkpoints.extend([
    ("p1_r003", seed03),
    ("p1_r012", seed12),
    ("old08_3_e_r02", old08_3_e02),
])

for variant in selected:
    root = variant_work_root(variant) / "global_checkpoints"
    for r in [1, 2, 3, PHASE2_ROUNDS_PER_VARIANT]:
        ckpt = root / f"phase2_round{r:03d}_global.pt"
        if ckpt.exists():
            checkpoints.append((f"{variant['name']}_r{r:03d}", ckpt))

print("eval_workspace:", EVAL_WORKSPACE)
print("checkpoints:")
for label, path in checkpoints:
    print(" ", label, path, "exists=", path.exists())

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--workspace",
    str(EVAL_WORKSPACE),
    "--splits",
    "total",
    "--batch-size",
    "16",
    "--no-plots",
    "--verbose",
]
for label, path in checkpoints:
    if path.exists():
        cmd.extend(["--checkpoint", f"{label}={path}"])

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

summary_csv = REPORT_DIR / "paper_protocol_eval_summary.csv"
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    out = REPORT_DIR / "paper_protocol_eval_summary_0832_early_total.csv"
    df.to_csv(out, index=False)
    display(df.sort_values(["split", "map50_95", "map50"], ascending=[True, False, False])[["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]])
    print("saved:", out)
else:
    print("No summary yet:", summary_csv)

eval_workspace: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_2_phase2_fedsto_dqa_sweep/a_fedsto_exact_r003
checkpoints:
  p1_r003 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round003_global.pt exists= True
  p1_r012 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round012_global.pt exists= True
  old08_3_e_r02 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_phase2_update_gating_sweep/e_ug_best_phase1_r003/global_checkpoints/phase2_round002_global.pt exists= True
  a_fedsto_exact_r003_r001 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_2_phase2_fedsto_dqa_sweep/a_fedsto_exact_r003/global_checkpoints/phase2_round001_global.pt exists= True
  a_fedsto_exact_r003_r002 /app/

,checkpoint_label,split,precision,recall,map50,map50_95
0,p1_r003,scene_total,0.471,0.401,0.375,0.204
1,p1_r012,scene_total,NaN,NaN,NaN,NaN
2,old08_3_e_r02,scene_total,NaN,NaN,NaN,NaN
3,a_fedsto_exact_r003_r001,scene_total,NaN,NaN,NaN,NaN
4,a_fedsto_exact_r003_r002,scene_total,NaN,NaN,NaN,NaN
5,a_fedsto_exact_r003_r003,scene_total,NaN,NaN,NaN,NaN
6,a_fedsto_exact_r003_r010,scene_total,NaN,NaN,NaN,NaN
7,b_fedsto_dqa_residual_r003_r001,scene_total,NaN,NaN,NaN,NaN
8,b_fedsto_dqa_residual_r003_r002,scene_total,NaN,NaN,NaN,NaN
9,b_fedsto_dqa_residual_r003_r003,scene_total,NaN,NaN,NaN,NaN


saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_2_phase2_fedsto_dqa_sweep/a_fedsto_exact_r003/validation_reports/paper_protocol_eval_summary_0832_early_total.csv


In [7]:
# 4 split final evaluation。上の early-total で良さそうな checkpoint をここに追加して比較。
BEST_LABELS = []  # 例: ["c_dqa_high_light_head_r003_r002", "b_fedsto_dqa_residual_r003_r003"]

ckpt_map = {label: path for label, path in checkpoints if path.exists()}
if not BEST_LABELS:
    BEST_LABELS = [label for label in ckpt_map if label.endswith(f"r{PHASE2_ROUNDS_PER_VARIANT:03d}")]

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--workspace",
    str(EVAL_WORKSPACE),
    "--splits",
    "highway,citystreet,residential,total",
    "--batch-size",
    "16",
    "--no-plots",
    "--verbose",
]
for label in ["p1_r003", "p1_r012", "old08_3_e_r02", *BEST_LABELS]:
    path = ckpt_map.get(label)
    if path and path.exists():
        cmd.extend(["--checkpoint", f"{label}={path}"])

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

summary_csv = REPORT_DIR / "paper_protocol_eval_summary.csv"
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    out = REPORT_DIR / "paper_protocol_eval_summary_0832_selected_4splits.csv"
    df.to_csv(out, index=False)
    display(df.sort_values(["split", "map50_95", "map50"], ascending=[True, False, False])[["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]])
    print("saved:", out)

/root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py --workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_2_phase2_fedsto_dqa_sweep/a_fedsto_exact_r003 --splits highway,citystreet,residential,total --batch-size 16 --no-plots --verbose --checkpoint p1_r003=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round003_global.pt --checkpoint p1_r012=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round012_global.pt --checkpoint old08_3_e_r02=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_phase2_update_gating_sweep/e_ug_best_phase1_r003/global_checkpoints/phase2_round002_global.pt --checkpoint a_fedsto_exact_r003_r010=/app/Object_Detectio

Traceback (most recent call last):
  File "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py", line 47, in <module>
    raise SystemExit(main())
  File "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py", line 43, in main
    return shared.main(args)
  File "/app/Object_Detection/navigating_data_heterogeneity/evaluate_paper_protocol.py", line 478, in main
    report_root, rows, class_rows, val_python = run_evaluations(setup, workspace, checkpoints, split_specs, args)
  File "/app/Object_Detection/navigating_data_heterogeneity/evaluate_paper_protocol.py", line 373, in run_evaluations
    val_python = select_val_python(args.python_executable)
  File "/app/Object_Detection/navigating_data_heterogeneity/evaluate_paper_protocol.py", line 203, in select_val_python
    raise RuntimeError(
RuntimeError: Could not find a Python interpreter with EfficientTeacher validation dependencies. Tried: /root/micromamba/

CalledProcessError: Command '['/root/micromamba/envs/al_yolov8/bin/python', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py', '--workspace', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_2_phase2_fedsto_dqa_sweep/a_fedsto_exact_r003', '--splits', 'highway,citystreet,residential,total', '--batch-size', '16', '--no-plots', '--verbose', '--checkpoint', 'p1_r003=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round003_global.pt', '--checkpoint', 'p1_r012=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round012_global.pt', '--checkpoint', 'old08_3_e_r02=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_phase2_update_gating_sweep/e_ug_best_phase1_r003/global_checkpoints/phase2_r

In [ ]:
# pseudoGT / DQA gate 推移を確認するセル。
rows = []
for variant in selected:
    stats_root = variant_stats_root(variant)
    for r in range(1, PHASE2_ROUNDS_PER_VARIANT + 1):
        path = stats_root / f"phase2_round{r:03d}.json"
        if not path.exists():
            continue
        data = json.loads(path.read_text())
        counts = [0.0] * 10
        qsum = [0.0] * 10
        for client in data.get("clients", []):
            for i, value in enumerate(client.get("counts", [])):
                counts[i] += float(value)
            for i, value in enumerate(client.get("quality_sums", [])):
                qsum[i] += float(value)
        total = sum(counts)
        rows.append({
            "variant": variant["name"],
            "round": r,
            "pseudo_total": total,
            "mean_quality": sum(qsum) / total if total else None,
            "active_classes": sum(1 for x in counts if x > 0),
            "car_count": counts[2],
            "person_count": counts[0],
            "traffic_sign_count": counts[8],
        })

stats_df = pd.DataFrame(rows)
if not stats_df.empty:
    display(stats_df)
    display(stats_df.groupby("variant")[["pseudo_total", "mean_quality", "active_classes"]].agg(["min", "max", "mean"]))
else:
    print("No pseudo stats yet")

for variant in selected:
    state_path = variant_work_root(variant) / "dqa_cwa_state.json"
    if not state_path.exists():
        continue
    state = json.loads(state_path.read_text())
    gate = state.get("phase2_fedsto_dqa_sweep", [])
    if gate:
        print("\n", variant["name"])
        display(pd.DataFrame(gate))

## 期待する読み方

FedSTO exact が 08_3 DQA を超えるなら、今までの DQA はやはり **phase2 で head/bbox/cls を動かさなさすぎ**です。

`b_fedsto_dqa_residual_r003` が FedSTO exact を超えれば、FedSTO の client objective は良く、**集約だけ DQA にする価値がある**ことになります。

`c` or `d` が一番良ければ、FedSTO の full update は強すぎて、**high-quality pseudoGT だけ bbox/cls を薄く使う**のが正解に近いです。

`e` が良ければ、問題の中心は pseudoGT loss より **non-IID client drift の集約**です。逆に `e` が伸びないなら、server anchor を強くしすぎて target 情報を殺している可能性が高いです。